# Day 077 — Exercise 1: open_camera and read_frame

**What you'll build:** The two camera primitives.

**Why it matters:** Every function in `live_vision.py` builds on these. The `camera_fn` injection means the entire pipeline runs in a headless test environment without a real camera.

In [ ]:
import numpy as np
from PIL import Image as _PILImage

def _make_mock_frame(h=100, w=100, val=50):
    return np.full((h, w, 3), val, dtype=np.uint8)

class _MockCap:
    def __init__(self, n=5, h=100, w=100):
        self._frames = [_make_mock_frame(h, w) for _ in range(n)]
        self._idx = 0
    def isOpened(self):
        return True
    def read(self):
        if self._idx >= len(self._frames):
            return False, None
        f = self._frames[self._idx]; self._idx += 1
        return True, f
    def release(self):
        pass
    def get(self, prop):
        return 0.0

_mock_camera_fn = lambda device: _MockCap(n=5)
_mock_analyze_fn = lambda img, q: 'FRAME:' + q[:12]


## Task

1. `open_camera(device=0, camera_fn=None) -> cap`
   - If `camera_fn` is not None: `return camera_fn(device)`
   - Else: `import cv2; cap = cv2.VideoCapture(device)`; raise `RuntimeError` if `not cap.isOpened()`; return cap

2. `read_frame(cap) -> (bool, ndarray|None)`
   - One line: `return cap.read()`

## Your Implementation

In [ ]:
def open_camera(device=0, camera_fn=None):
    """Open a camera device and return a VideoCapture-compatible object."""
    raise NotImplementedError

def read_frame(cap):
    """Read one frame. Returns (success: bool, frame: ndarray|None)."""
    raise NotImplementedError


In [ ]:
def open_camera(device=0, camera_fn=None):
    if camera_fn is not None:
        return camera_fn(device)
    import cv2
    cap = cv2.VideoCapture(device)
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open camera device {device}')
    return cap

def read_frame(cap):
    return cap.read()


## Automated checks

In [ ]:

score, total = 0, 4
try:
    cap = open_camera(device=0, camera_fn=_mock_camera_fn)
    assert cap is not None and cap.isOpened()
    score += 1; print("✅ open_camera returns cap via camera_fn")

    device_seen = []
    def _dfn(d): device_seen.append(d); return _MockCap(n=2)
    open_camera(device=99, camera_fn=_dfn)
    assert device_seen == [99], f"device not forwarded: {device_seen}"
    score += 1; print("✅ device argument forwarded to camera_fn")

    cap2 = _MockCap(n=2)
    ret, frame = read_frame(cap2)
    assert ret is True and frame is not None
    score += 1; print("✅ read_frame returns (True, frame) on valid cap")

    cap3 = _MockCap(n=1)
    read_frame(cap3)
    ret2, frame2 = read_frame(cap3)
    assert ret2 is False and frame2 is None
    score += 1; print("✅ read_frame returns (False, None) when exhausted")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def open_camera(device=0, camera_fn=None):
    if camera_fn is not None:
        return camera_fn(device)
    import cv2
    cap = cv2.VideoCapture(device)
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open camera device {device}')
    return cap

def read_frame(cap):
    return cap.read()
```

**Why import cv2 inside the else branch?** Avoids import errors on machines without opencv installed. The module loads cleanly in any environment; cv2 is only imported when actually needed.

</details>